# Data Pipeline and Exploratory Data Analysis

## Setup

In [ ]:
import random, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import hashlib

sys.path.append(str(Path.cwd()))
from shared_class_labels import build_index, report, CLASS_NAMES, N_CLASSES

SEED = 42
val_fraction, test_fraction = 0.15, 0.15
LAB, FIELD = "#0033ff", "#ff0000" # blue = laboratory, red = field
random.seed(SEED)

index = build_index()
report(index)

## Shared class labels
The two datasets have different naming conventions for the same classes. This list is used to link the two datasets to each other based on the class names. The shared_class_labels.py script is used to group the datasets according to these classes.

In [ ]:
SHARED_CLASSES: dict[str, tuple[str, str]] = {
    "apple_scab": ("Apple___Apple_scab", "Apple Scab Leaf"),
    "apple_rust": ("Apple___Cedar_apple_rust", "Apple rust leaf"),
    "apple_healthy": ("Apple___healthy", "Apple leaf"),
    "blueberry_healthy": ("Blueberry___healthy", "Blueberry leaf"),
    "cherry_healthy": ("Cherry_(including_sour)___healthy", "Cherry leaf"),
    "corn_gray_spot": ("Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn Gray leaf spot"),
    "corn_rust": ("Corn_(maize)___Common_rust_", "Corn rust leaf"),
    "corn_blight": ("Corn_(maize)___Northern_Leaf_Blight", "Corn leaf blight"),
    "grape_black_rot": ("Grape___Black_rot", "grape leaf black rot"),
    "grape_healthy": ("Grape___healthy", "grape leaf"),
    "peach_healthy": ("Peach___healthy", "Peach leaf"),
    "pepper_healthy": ("Pepper,_bell___healthy", "Bell_pepper leaf"),
    "potato_early_blight": ("Potato___Early_blight", "Potato leaf early blight"),
    "potato_late_blight": ("Potato___Late_blight", "Potato leaf late blight"),
    "raspberry_healthy": ("Raspberry___healthy", "Raspberry leaf"),
    "soybean_healthy": ("Soybean___healthy", "Soyabean leaf"),
    "squash_mildew": ("Squash___Powdery_mildew", "Squash Powdery mildew leaf"),
    "strawberry_healthy": ("Strawberry___healthy", "Strawberry leaf"),
    "tomato_bacterial_spot": ("Tomato___Bacterial_spot", "Tomato leaf bacterial spot"),
    "tomato_early_blight": ("Tomato___Early_blight", "Tomato Early blight leaf"),
    "tomato_late_blight": ("Tomato___Late_blight", "Tomato leaf late blight"),
    "tomato_leaf_mold": ("Tomato___Leaf_Mold", "Tomato mold leaf"),
    "tomato_septoria": ("Tomato___Septoria_leaf_spot", "Tomato Septoria leaf spot"),
    "tomato_yellow_virus": ("Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato leaf yellow virus"),
    "tomato_mosaic_virus": ("Tomato___Tomato_mosaic_virus", "Tomato leaf mosaic virus"),
    "tomato_healthy": ("Tomato___healthy", "Tomato leaf"),
}

CLASS_NAMES = list(SHARED_CLASSES)
N_CLASSES = len(CLASS_NAMES)
print(f"{N_CLASSES} number of classes shared between PlantVillage (lab-condition) and PlantDoc (field-condition)")

## Exploratory Data Analysis

In [ ]:
df = pd.DataFrame(
    [(str(path), domain, CLASS_NAMES[label])
     for domain, pairs in (("lab", index["lab"]), ("field", index["field"]))
     for path, label in pairs],
    columns=["path", "domain", "class"])

print(f"{len(df):,} images")
df.head()

In [ ]:
first = {d: {} for d in ("lab", "field")}
for path, domain, cls in df[["path", "domain", "class"]].itertuples(index=False):
    first[domain].setdefault(cls, path)

show = ["apple_scab", "corn_rust", "grape_black_rot", "tomato_early_blight", "squash_mildew"]

fig, axes = plt.subplots(2, len(show), figsize=(3 * len(show), 6))
for col, cls in enumerate(show):
    for row, (domain, colour) in enumerate((("lab", LAB), ("field", FIELD))):
        ax = axes[row, col]
        ax.set_xticks([]); ax.set_yticks([])
        ax.imshow(Image.open(first[domain][cls]).convert("RGB"))
        if col == 0:
            ax.set_ylabel(domain.upper(), fontsize=11, fontweight="bold", color=colour)
        if row == 0:
            ax.set_title(cls.replace("_", " "), fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# Class balance 
counts = df.pivot_table(index="class", columns="domain", aggfunc="size").reindex(CLASS_NAMES)

table = pd.DataFrame({
    "lab": counts.lab,
    "lab %": (100 * counts.lab / counts.lab.sum()).round(1),
    "field": counts.field,
    "field %": (100 * counts.field / counts.field.sum()).round(1),
    "lab:field": (counts.lab / counts.field).round(1),
})
table.loc["TOTAL"] = [counts.lab.sum(), 100.0, counts.field.sum(), 100.0,
                      round(counts.lab.sum() / counts.field.sum(), 1)]
table[["lab", "field"]] = table[["lab", "field"]].astype(int)

print(f"lab    {counts.lab.min():,}-{counts.lab.max():,} per class "
      f"(median {counts.lab.median():,.0f}), {counts.lab.max()/counts.lab.min():.0f}x imbalance")
print(f"field  {counts.field.min()}-{counts.field.max()} per class "
      f"(median {counts.field.median():.0f}), {counts.field.max()/counts.field.min():.0f}x imbalance")

display(table)

In [ ]:
# Image sizes
sizes = []
for path in df["path"]:
    with Image.open(path) as im:
        sizes.append(im.size)
df["width"], df["height"] = zip(*sizes)

for domain in ("lab", "field"):
    g = df[df.domain == domain]
    area = (g.width * g.height).median()
    print(f"{domain:<6} median {g.width.median():.0f}x{g.height.median():.0f}"
          f"{len(set(zip(g.width, g.height))):,} distinct sizes"
          f"{area / (224*224):.1f}x downsampling to resize to 224x224") # only if we want to resize - this will distort the pictures that are rectangular

In [ ]:
# Duplicates filtering
def phash(path):
    with Image.open(path) as im:
        im.draft("L", (32, 32))   
        a = np.asarray(im.convert("L").resize((8, 8)), dtype=float)
    return int("".join("1" if b else "0" for b in (a > a.mean()).flatten()), 2)

df["hash"] = [phash(p) for p in df["path"]]
df["group"] = df.domain + "_" + df["class"] + "_" + df["hash"].astype(str)

lab = df[df.domain == "lab"]
groups = lab.groupby("group").size()
print(f"{len(lab):,} lab images -> {len(groups):,} distinct groups")
print(f"{(groups > 1).sum():,} groups contain duplicates; "
      f"largest group holds {groups.max()} images")
print(f"{1 - len(groups) / len(lab):.1%} of the lab set is a near-duplicate of another image")

## Leakage check

In [ ]:
lab_hashes = set(df.loc[df.domain == "lab", "hash"])
leaks = df[(df.domain == "field") & df["hash"].isin(lab_hashes)]
print(f"lab/field collisions: {len(leaks)}")
if len(leaks):
    print(leaks[["path", "class"]].to_string(index=False))

## Final data split

In [ ]:
df["split"] = "field_test"  # all field images are in the test split, lab images are also added below
targets = {"train": 1 - val_fraction - test_fraction, "val": val_fraction, "test": test_fraction}

for cls in CLASS_NAMES:
    rows = df[(df.domain == "lab") & (df["class"] == cls)]
    groups = list(rows.groupby("group").groups.values())
    random.Random(f"{SEED}-{cls}").shuffle(groups)   # seeded per class
    groups.sort(key=len, reverse=True)
    have = dict.fromkeys(targets, 0)
    for idx in groups:
        short = min(have, key=lambda s: have[s] - targets[s] * len(rows))
        df.loc[idx, "split"] = short
        have[short] += len(idx)

print(pd.crosstab(df.domain, df.split).to_string())

missing = [(c, s) for c in CLASS_NAMES for s in targets
           if not len(df[(df.domain == "lab") & (df["class"] == c) & (df.split == s)])]
print("\nlab classes missing from a split:", missing or "none")

In [ ]:
ROOT = Path.cwd().parent

split_df = df[["path", "domain", "class", "split", "group"]].copy()
split_df["path"] = [str(Path(p).relative_to(ROOT)) for p in split_df["path"]]

out = ROOT / "data" / "data_split.csv"
split_df.to_csv(out, index=False)

print(f"saved {out}")